In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os

PROJECT_DIR = Path("/content/drive/MyDrive/master/courses/AI_in_medicine/AI_MD_Project")
os.chdir(PROJECT_DIR)

print("PWD:", Path.cwd())

Path("src/retrieval").mkdir(parents=True, exist_ok=True)
Path("docs").mkdir(parents=True, exist_ok=True)
Path("outputs").mkdir(parents=True, exist_ok=True)

Path("src/retrieval/__init__.py").touch()

In [ ]:
from pathlib import Path

required_files = [
    "data/processed/phq8_item_dataset_with_splits.csv",
    "data/processed/utterance_bank.json",
    "src/data/dataset_loader.py",
]

for path in required_files:
    print(path, "->", Path(path).exists())

In [ ]:
!pip -q install scikit-learn rank-bm25

**PA9 - create src/retrieval/tfidf_retriever.py**

In [ ]:
%%writefile src/retrieval/tfidf_retriever.py
"""
TF-IDF based utterance retriever.

Given a PHQ-8 item text and a participant's utterances, return the top-k
utterances that are most similar to the item text.
"""

from typing import List
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def clean_for_retrieval(text: str) -> str:
    if text is None:
        return ""

    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def retrieve_top_k_tfidf(
    item_text: str,
    utterances: List[str],
    k: int = 5,
) -> List[str]:
    """
    Retrieve top-k utterances using TF-IDF cosine similarity.

    Parameters
    ----------
    item_text : str
        PHQ-8 item text.
    utterances : list[str]
        Candidate utterances for one participant.
    k : int
        Number of utterances to return.

    Returns
    -------
    list[str]
        Top-k utterances, ordered from most to least relevant.
    """
    if utterances is None:
        return []

    utterances = [
        str(u).strip()
        for u in utterances
        if u is not None and str(u).strip() != ""
    ]

    if len(utterances) == 0:
        return []

    if k <= 0:
        return []

    if item_text is None or str(item_text).strip() == "":
        return utterances[:k]

    cleaned_query = clean_for_retrieval(item_text)
    cleaned_utterances = [clean_for_retrieval(u) for u in utterances]

    valid_pairs = [
        (original, cleaned)
        for original, cleaned in zip(utterances, cleaned_utterances)
        if cleaned != ""
    ]

    if len(valid_pairs) == 0:
        return utterances[:k]

    valid_utterances = [p[0] for p in valid_pairs]
    valid_cleaned = [p[1] for p in valid_pairs]

    documents = [cleaned_query] + valid_cleaned

    try:
        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            min_df=1,
        )
        tfidf = vectorizer.fit_transform(documents)

        query_vec = tfidf[0]
        utterance_vecs = tfidf[1:]

        scores = cosine_similarity(query_vec, utterance_vecs).flatten()

    except ValueError:
        # Happens if vocabulary is empty.
        return valid_utterances[:k]

    ranked_indices = scores.argsort()[::-1]
    top_indices = ranked_indices[: min(k, len(valid_utterances))]

    return [valid_utterances[i] for i in top_indices]


if __name__ == "__main__":
    item_text = "Trouble falling or staying asleep, or sleeping too much"

    utterances = [
        "I cannot fall asleep until 3 AM.",
        "I went to the grocery store yesterday.",
        "I wake up many times during the night.",
        "My appetite is normal.",
    ]

    print(retrieve_top_k_tfidf(item_text, utterances, k=2))

In [ ]:
!python src/retrieval/tfidf_retriever.py

**PA10- create src/retrieval/bm25_retriever.py**

In [ ]:
%pip install rank-bm25

In [ ]:
%%writefile src/retrieval/bm25_retriever.py
"""
BM25 based utterance retriever.

Given a PHQ-8 item text and a participant's utterances, return the top-k
utterances that are most relevant according to BM25.
"""

from typing import List
import re

from rank_bm25 import BM25Okapi


def tokenize(text: str) -> list[str]:
    if text is None:
        return []

    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    if text == "":
        return []

    return text.split()


def retrieve_top_k_bm25(
    item_text: str,
    utterances: List[str],
    k: int = 5,
) -> List[str]:
    """
    Retrieve top-k utterances using BM25.

    Parameters
    ----------
    item_text : str
        PHQ-8 item text.
    utterances : list[str]
        Candidate utterances for one participant.
    k : int
        Number of utterances to return.

    Returns
    -------
    list[str]
        Top-k utterances, ordered from most to least relevant.
    """
    if utterances is None:
        return []

    utterances = [
        str(u).strip()
        for u in utterances
        if u is not None and str(u).strip() != ""
    ]

    if len(utterances) == 0:
        return []

    if k <= 0:
        return []

    query_tokens = tokenize(item_text)

    if len(query_tokens) == 0:
        return utterances[:k]

    tokenized_utterances = [tokenize(u) for u in utterances]

    valid_pairs = [
        (original, tokens)
        for original, tokens in zip(utterances, tokenized_utterances)
        if len(tokens) > 0
    ]

    if len(valid_pairs) == 0:
        return utterances[:k]

    valid_utterances = [p[0] for p in valid_pairs]
    valid_tokens = [p[1] for p in valid_pairs]

    bm25 = BM25Okapi(valid_tokens)
    scores = bm25.get_scores(query_tokens)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True,
    )

    top_indices = ranked_indices[: min(k, len(valid_utterances))]

    return [valid_utterances[i] for i in top_indices]


if __name__ == "__main__":
    item_text = "Trouble falling or staying asleep, or sleeping too much"

    utterances = [
        "I cannot fall asleep until 3 AM.",
        "I went to the grocery store yesterday.",
        "I wake up many times during the night.",
        "My appetite is normal.",
    ]

    print(retrieve_top_k_bm25(item_text, utterances, k=2))

In [ ]:
!python src/retrieval/bm25_retriever.py

**PA11 + PA12- Creating a dataset with retrieval and baseline**

In [ ]:
%%writefile src/retrieval/build_retrieval_dataset.py
"""
Build retrieval-enhanced PHQ-8 item-level datasets.

Inputs:
- data/processed/phq8_item_dataset_with_splits.csv
- data/processed/utterance_bank.json

Outputs:
- data/processed/phq8_item_dataset_with_retrieval.csv
- data/processed/phq8_item_dataset_full.csv

New columns:
- retrieved_utterances
- baseline_utterances
- n_available_utterances
- n_retrieved_utterances
- n_baseline_utterances
"""

from pathlib import Path
import argparse
import hashlib
import json
import random
import sys

import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parents[2]
sys.path.append(str(PROJECT_ROOT))

from src.data.dataset_loader import load_item_dataset
from src.retrieval.tfidf_retriever import retrieve_top_k_tfidf
from src.retrieval.bm25_retriever import retrieve_top_k_bm25


UTT_SEP = " [UTT_SEP] "


def normalize_participant_id(value):
    if value is None or pd.isna(value):
        return None

    text = str(value).strip()

    if text.endswith(".0"):
        text = text[:-2]

    return text


def load_utterance_bank(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"utterance_bank.json does not exist: {path}")

    with open(path, "r", encoding="utf-8") as f:
        utterance_bank = json.load(f)

    if not isinstance(utterance_bank, dict):
        raise ValueError("utterance_bank must be a dictionary.")

    normalized = {}

    for pid, utterances in utterance_bank.items():
        pid = normalize_participant_id(pid)

        if not isinstance(utterances, list):
            raise ValueError(f"Utterances for participant {pid} must be a list.")

        cleaned = [
            str(u).strip()
            for u in utterances
            if u is not None and str(u).strip() != ""
        ]

        normalized[pid] = cleaned

    return normalized


def choose_retrieved_utterances(item_text, utterances, method, k):
    if method == "tfidf":
        return retrieve_top_k_tfidf(item_text, utterances, k=k)

    if method == "bm25":
        return retrieve_top_k_bm25(item_text, utterances, k=k)

    raise ValueError(f"Unknown retrieval method: {method}")


def stable_random_sample(utterances, k, seed_text):
    if len(utterances) <= k:
        return utterances

    seed_hash = hashlib.md5(seed_text.encode("utf-8")).hexdigest()
    seed_int = int(seed_hash[:8], 16)

    rng = random.Random(seed_int)
    indices = list(range(len(utterances)))
    rng.shuffle(indices)

    selected_indices = sorted(indices[:k])

    return [utterances[i] for i in selected_indices]


def choose_baseline_utterances(utterances, k, strategy, seed_text):
    if utterances is None:
        return []

    utterances = [
        str(u).strip()
        for u in utterances
        if u is not None and str(u).strip() != ""
    ]

    if len(utterances) == 0:
        return []

    if k <= 0:
        return []

    if strategy == "first":
        return utterances[:k]

    if strategy == "random":
        return stable_random_sample(utterances, k=k, seed_text=seed_text)

    raise ValueError(f"Unknown baseline strategy: {strategy}")


def join_utterances(utterances):
    utterances = [
        str(u).strip()
        for u in utterances
        if u is not None and str(u).strip() != ""
    ]

    return UTT_SEP.join(utterances)


def validate_retrieval_dataset(df):
    required_cols = [
        "participant_id",
        "item_id",
        "item_name",
        "item_text",
        "label",
        "transcript_text",
        "split",
        "retrieved_utterances",
        "baseline_utterances",
        "n_available_utterances",
        "n_retrieved_utterances",
        "n_baseline_utterances",
    ]

    missing = [col for col in required_cols if col not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if df.empty:
        raise ValueError("Retrieval dataset is empty.")

    invalid_retrieval = df[
        (df["n_available_utterances"] > 0) &
        (df["n_retrieved_utterances"] == 0)
    ]

    if len(invalid_retrieval) > 0:
        raise ValueError(
            "Some rows have available utterances but no retrieved utterances. "
            f"Examples:\n{invalid_retrieval[['participant_id', 'item_id']].head()}"
        )

    invalid_baseline = df[
        (df["n_available_utterances"] > 0) &
        (df["n_baseline_utterances"] == 0)
    ]

    if len(invalid_baseline) > 0:
        raise ValueError(
            "Some rows have available utterances but no baseline utterances. "
            f"Examples:\n{invalid_baseline[['participant_id', 'item_id']].head()}"
        )

    return True


def build_retrieval_dataset(
    dataset_path: Path,
    utterance_bank_path: Path,
    retrieval_output_path: Path,
    full_output_path: Path,
    method: str,
    k: int,
    baseline_strategy: str,
):
    df = load_item_dataset(dataset_path)
    utterance_bank = load_utterance_bank(utterance_bank_path)

    df = df.copy()
    df["participant_id"] = df["participant_id"].map(normalize_participant_id)

    retrieved_values = []
    baseline_values = []

    n_available = []
    n_retrieved = []
    n_baseline = []

    missing_participants = set()

    for _, row in df.iterrows():
        participant_id = normalize_participant_id(row["participant_id"])
        item_id = int(row["item_id"])
        item_text = row["item_text"]

        utterances = utterance_bank.get(participant_id, [])

        if len(utterances) == 0:
            missing_participants.add(participant_id)

        retrieved = choose_retrieved_utterances(
            item_text=item_text,
            utterances=utterances,
            method=method,
            k=k,
        )

        seed_text = f"{participant_id}_{item_id}_{baseline_strategy}"

        baseline = choose_baseline_utterances(
            utterances=utterances,
            k=k,
            strategy=baseline_strategy,
            seed_text=seed_text,
        )

        retrieved_values.append(join_utterances(retrieved))
        baseline_values.append(join_utterances(baseline))

        n_available.append(len(utterances))
        n_retrieved.append(len(retrieved))
        n_baseline.append(len(baseline))

    df["retrieved_utterances"] = retrieved_values
    df["baseline_utterances"] = baseline_values

    df["n_available_utterances"] = n_available
    df["n_retrieved_utterances"] = n_retrieved
    df["n_baseline_utterances"] = n_baseline

    validate_retrieval_dataset(df)

    retrieval_output_path.parent.mkdir(parents=True, exist_ok=True)
    full_output_path.parent.mkdir(parents=True, exist_ok=True)

    retrieval_cols = [
        col for col in df.columns
        if col != "baseline_utterances" and col != "n_baseline_utterances"
    ]

    df[retrieval_cols].to_csv(retrieval_output_path, index=False)
    df.to_csv(full_output_path, index=False)

    print("Retrieval datasets created successfully.")
    print(f"Retrieval method: {method}")
    print(f"k: {k}")
    print(f"Baseline strategy: {baseline_strategy}")

    print(f"\nSaved retrieval dataset:")
    print(retrieval_output_path)

    print(f"\nSaved full dataset:")
    print(full_output_path)

    print("\nDataset shape:")
    print(df.shape)

    print("\nRows per split:")
    print(df["split"].value_counts())

    print("\nParticipants missing from utterance_bank:")
    print(len(missing_participants))

    print("\nRetrieved utterance counts:")
    print(df["n_retrieved_utterances"].describe())

    print("\nBaseline utterance counts:")
    print(df["n_baseline_utterances"].describe())

    return df


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--dataset_path",
        type=str,
        default="data/processed/phq8_item_dataset_with_splits.csv",
    )

    parser.add_argument(
        "--utterance_bank_path",
        type=str,
        default="data/processed/utterance_bank.json",
    )

    parser.add_argument(
        "--retrieval_output_path",
        type=str,
        default="data/processed/phq8_item_dataset_with_retrieval.csv",
    )

    parser.add_argument(
        "--full_output_path",
        type=str,
        default="data/processed/phq8_item_dataset_full.csv",
    )

    parser.add_argument(
        "--method",
        type=str,
        default="tfidf",
        choices=["tfidf", "bm25"],
    )

    parser.add_argument(
        "--k",
        type=int,
        default=5,
    )

    parser.add_argument(
        "--baseline_strategy",
        type=str,
        default="first",
        choices=["first", "random"],
    )

    return parser.parse_args()


def main():
    args = parse_args()

    build_retrieval_dataset(
        dataset_path=Path(args.dataset_path),
        utterance_bank_path=Path(args.utterance_bank_path),
        retrieval_output_path=Path(args.retrieval_output_path),
        full_output_path=Path(args.full_output_path),
        method=args.method,
        k=args.k,
        baseline_strategy=args.baseline_strategy,
    )


if __name__ == "__main__":
    main()

In [ ]:
!python src/retrieval/build_retrieval_dataset.py \
  --dataset_path data/processed/phq8_item_dataset_with_splits.csv \
  --utterance_bank_path data/processed/utterance_bank.json \
  --retrieval_output_path data/processed/phq8_item_dataset_with_retrieval.csv \
  --full_output_path data/processed/phq8_item_dataset_full.csv \
  --method tfidf \
  --k 5 \
  --baseline_strategy first

In [ ]:
!python src/retrieval/build_retrieval_dataset.py \
  --dataset_path data/processed/phq8_item_dataset_with_splits.csv \
  --utterance_bank_path data/processed/utterance_bank.json \
  --retrieval_output_path data/processed/phq8_item_dataset_with_retrieval_bm25.csv \
  --full_output_path data/processed/phq8_item_dataset_full_bm25.csv \
  --method bm25 \
  --k 5 \
  --baseline_strategy first

In [ ]:
import pandas as pd

df_full = pd.read_csv("data/processed/phq8_item_dataset_full.csv")

print(df_full.shape)
display(df_full.head())

print(df_full[[
    "participant_id",
    "item_id",
    "item_name",
    "split",
    "n_available_utterances",
    "n_retrieved_utterances",
    "n_baseline_utterances",
]].head())

print("Missing retrieved:", df_full["retrieved_utterances"].isna().sum())
print("Missing baseline:", df_full["baseline_utterances"].isna().sum())
print("Splits:")
print(df_full["split"].value_counts())

In [ ]:
from src.data.dataset_loader import load_item_dataset

df_loaded = load_item_dataset("data/processed/phq8_item_dataset_full.csv")

print(df_loaded.shape)
print("retrieved_utterances" in df_loaded.columns)
print("baseline_utterances" in df_loaded.columns)

**PA13 - Creating docs/retrieval examples.md**

In [ ]:
%%writefile src/retrieval/export_retrieval_examples.py
"""
Export qualitative retrieval examples to docs/retrieval_examples.md.

The output is intended for manual review.
"""

from pathlib import Path
import argparse
import random

import pandas as pd


UTT_SEP = " [UTT_SEP] "


PREFERRED_ITEMS = [
    "Sleep",
    "Appetite",
    "Concentrating",
    "Tired",
]


def split_utterance_string(text):
    if text is None or pd.isna(text):
        return []

    text = str(text).strip()

    if text == "":
        return []

    return [
        part.strip()
        for part in text.split(UTT_SEP)
        if part.strip() != ""
    ]


def select_examples(df, n_examples, seed):
    rng = random.Random(seed)

    selected_rows = []

    # Prefer clinically interesting / harder items first.
    for item_name in PREFERRED_ITEMS:
        item_df = df[df["item_name"] == item_name]

        if len(item_df) == 0:
            continue

        # Prefer validation/test examples for qualitative inspection.
        preferred_df = item_df[item_df["split"].isin(["validation", "test"])]

        if len(preferred_df) == 0:
            preferred_df = item_df

        sample_n = min(3, len(preferred_df))
        indices = list(preferred_df.index)
        rng.shuffle(indices)

        selected_rows.extend(indices[:sample_n])

    # Fill remaining examples randomly.
    if len(selected_rows) < n_examples:
        remaining_indices = [
            idx for idx in df.index
            if idx not in selected_rows
        ]

        rng.shuffle(remaining_indices)
        selected_rows.extend(remaining_indices[: n_examples - len(selected_rows)])

    selected_rows = selected_rows[:n_examples]

    return df.loc[selected_rows].copy()


def write_examples_markdown(df_examples, output_path):
    lines = []

    lines.append("# Retrieval Examples")
    lines.append("")
    lines.append("Manual qualitative review of retrieved utterances.")
    lines.append("")
    lines.append("Use the `comment` field to mark each example as `good`, `mixed`, or `bad`.")
    lines.append("")

    for i, (_, row) in enumerate(df_examples.iterrows(), start=1):
        retrieved = split_utterance_string(row["retrieved_utterances"])
        baseline = split_utterance_string(row["baseline_utterances"])

        lines.append(f"## Example {i}")
        lines.append("")
        lines.append(f"- participant_id: `{row['participant_id']}`")
        lines.append(f"- split: `{row['split']}`")
        lines.append(f"- item_id: `{row['item_id']}`")
        lines.append(f"- item_name: `{row['item_name']}`")
        lines.append(f"- label: `{row['label']}`")
        lines.append("")
        lines.append("### Item text")
        lines.append("")
        lines.append(str(row["item_text"]))
        lines.append("")
        lines.append("### Retrieved utterances")
        lines.append("")

        if retrieved:
            for utt in retrieved:
                lines.append(f"- {utt}")
        else:
            lines.append("- No retrieved utterances.")

        lines.append("")
        lines.append("### Baseline utterances")
        lines.append("")

        if baseline:
            for utt in baseline:
                lines.append(f"- {utt}")
        else:
            lines.append("- No baseline utterances.")

        lines.append("")
        lines.append("### Comment")
        lines.append("")
        lines.append("TODO: good / mixed / bad")
        lines.append("")
        lines.append("---")
        lines.append("")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines), encoding="utf-8")

    print("Retrieval examples saved successfully.")
    print(f"Output path: {output_path}")
    print(f"Examples: {len(df_examples)}")


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--input_path",
        type=str,
        default="data/processed/phq8_item_dataset_full.csv",
    )

    parser.add_argument(
        "--output_path",
        type=str,
        default="docs/retrieval_examples.md",
    )

    parser.add_argument(
        "--n_examples",
        type=int,
        default=12,
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=42,
    )

    return parser.parse_args()


def main():
    args = parse_args()

    input_path = Path(args.input_path)
    output_path = Path(args.output_path)

    if not input_path.exists():
        raise FileNotFoundError(f"Input file does not exist: {input_path}")

    df = pd.read_csv(input_path)

    required_cols = [
        "participant_id",
        "split",
        "item_id",
        "item_name",
        "item_text",
        "label",
        "retrieved_utterances",
        "baseline_utterances",
    ]

    missing = [col for col in required_cols if col not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df_examples = select_examples(
        df=df,
        n_examples=args.n_examples,
        seed=args.seed,
    )

    write_examples_markdown(df_examples, output_path)


if __name__ == "__main__":
    main()

In [ ]:
!python src/retrieval/export_retrieval_examples.py \
  --input_path data/processed/phq8_item_dataset_full.csv \
  --output_path docs/retrieval_examples.md \
  --n_examples 12

In [ ]:
!sed -n '1,180p' docs/retrieval_examples.md

In [ ]:
from pathlib import Path
import pandas as pd
import json

checks = {
    "tfidf_retriever.py": Path("src/retrieval/tfidf_retriever.py").exists(),
    "bm25_retriever.py": Path("src/retrieval/bm25_retriever.py").exists(),
    "build_retrieval_dataset.py": Path("src/retrieval/build_retrieval_dataset.py").exists(),
    "phq8_item_dataset_with_retrieval.csv": Path("data/processed/phq8_item_dataset_with_retrieval.csv").exists(),
    "phq8_item_dataset_full.csv": Path("data/processed/phq8_item_dataset_full.csv").exists(),
    "retrieval_examples.md": Path("docs/retrieval_examples.md").exists(),
}

for name, ok in checks.items():
    print(name, "->", ok)

df = pd.read_csv("data/processed/phq8_item_dataset_full.csv")

required_cols = [
    "participant_id",
    "item_id",
    "item_name",
    "item_text",
    "label",
    "transcript_text",
    "split",
    "retrieved_utterances",
    "baseline_utterances",
]

missing = [col for col in required_cols if col not in df.columns]
print("Missing required columns:", missing)

print("Shape:", df.shape)
print("Splits:")
print(df["split"].value_counts())

print("Retrieved counts:")
print(df["n_retrieved_utterances"].describe())

print("Baseline counts:")
print(df["n_baseline_utterances"].describe())

print("Rows with empty retrieved despite available utterances:")
print(len(df[(df["n_available_utterances"] > 0) & (df["n_retrieved_utterances"] == 0)]))